# Sesi 8 — Consuming & Presenting AI Output

**Demo instruktur (90 menit):** satu dataset retail sintetis menjadi tiga antarmuka keluaran: **visual** untuk perbandingan, **naratif** untuk pembaca bisnis, dan **rekomendasi terstruktur** untuk alur review. Jalankan sel secara berurutan dari atas.

| Bagian | Waktu | Yang perlu ditunjukkan |
|---|---:|---|
| Konteks dan kontrak output | 10 menit | Siapa pengguna tiap output? |
| Validasi data dan evidence | 20 menit | Angka dihitung Python |
| Visualisasi | 15 menit | Grafik cocok dengan evidence |
| Narasi LangChain | 20 menit | Satu panggilan API terstruktur |
| Validasi, review, diskusi | 25 menit | JSON valid belum berarti klaim benar atau aksi boleh dijalankan |

**Hasil belajar:** peserta dapat memilih format output menurut penggunanya, menelusuri klaim ke evidence, membedakan validasi skema dari validasi bisnis, dan menjelaskan mengapa rekomendasi memerlukan review manusia.

## 1. Skenario dan batas demo

Anda membantu tim analitik retail menjelaskan kinerja 24 transaksi sintetis. Manajer membutuhkan grafik, ringkasan, dan satu rekomendasi yang dapat diperiksa. Tidak ada refund, notifikasi, atau perubahan sistem yang akan dilakukan.

**Tanya kelas:** Jika kategori dengan profit terendah terlihat jelas, apa yang masih *belum* kita ketahui sebelum menyalahkan diskon?

> **Catatan produksi:** Ini demo edukasional. Sistem produksi memerlukan data aktual yang tervalidasi, otorisasi, jejak audit, pemantauan, dan proses persetujuan yang nyata.

In [ ]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, ValidationError

MODULE_DIR = "08-consuming-and-presenting-ai-output"
candidates = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (
        parent
        for parent in candidates
        if (parent / "modules" / MODULE_DIR / "project" / "data" / "retail_transactions.csv").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Dataset modul 8 tidak ditemukan. Jalankan Jupyter dari dalam repository ini."
    )

csv_path = repo_root / "modules" / MODULE_DIR / "project" / "data" / "retail_transactions.csv"
load_dotenv(repo_root / ".env", override=False)
print("Dataset:", csv_path.relative_to(repo_root))
print("API key siap:" if os.getenv("OPENAI_API_KEY") else "API key belum tersedia:",
      "ya" if os.getenv("OPENAI_API_KEY") else "isi OPENAI_API_KEY sebelum sel model")

## 2. Validasi sebelum analisis

Kontrak input meminta ID unik, tanggal yang dapat dibaca, kategori/wilayah yang terisi, angka valid, `sales` tidak negatif, dan `discount` antara 0 dan 1. Data kotor harus berhenti sebelum grafik atau model dibuat.

**Prediksi:** Apa akibat satu baris `discount=30` jika kita menganggap diskon selalu proporsi 0–1?

In [ ]:
REQUIRED_COLUMNS = {
    "order_id", "order_date", "region", "category", "sub_category",
    "sales", "profit", "discount",
}
TEXT_COLUMNS = ("order_id", "region", "category", "sub_category")
NUMBER_COLUMNS = ("sales", "profit", "discount")


def validate_retail_data(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    data.columns = (
        data.columns.str.strip().str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    missing = REQUIRED_COLUMNS - set(data.columns)
    if missing:
        raise ValueError(f"Kolom wajib hilang: {', '.join(sorted(missing))}")
    if data.empty:
        raise ValueError("Dataset tidak boleh kosong.")

    data = data[sorted(REQUIRED_COLUMNS)].copy()
    for column in TEXT_COLUMNS:
        data[column] = data[column].astype("string").str.strip()
    try:
        data["order_date"] = pd.to_datetime(data["order_date"], format="%Y-%m-%d", errors="raise")
        for column in NUMBER_COLUMNS:
            data[column] = pd.to_numeric(data[column], errors="raise")
    except (ValueError, TypeError) as exc:
        raise ValueError("Tanggal atau angka pada CSV tidak valid.") from exc

    if data[list(REQUIRED_COLUMNS)].isna().any(axis=None):
        raise ValueError("Kolom wajib tidak boleh berisi nilai kosong.")
    if data[list(TEXT_COLUMNS)].eq("").any(axis=None):
        raise ValueError("ID, kategori, subkategori, dan wilayah tidak boleh kosong.")
    if data["order_id"].duplicated().any():
        raise ValueError("order_id harus unik.")
    if (data["sales"] < 0).any():
        raise ValueError("sales tidak boleh negatif.")
    if (~data["discount"].between(0, 1)).any():
        raise ValueError("discount harus berada di antara 0 dan 1.")
    return data


def load_retail_data(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"CSV tidak ditemukan: {path}")
    return validate_retail_data(pd.read_csv(path))


data = load_retail_data(csv_path)
print(f"Valid: {len(data)} transaksi; {data['order_id'].nunique()} ID unik")
display(data.head())

## 3. Evidence dihitung secara deterministik

`profit_margin = total_profit / total_sales`. Nilai per kategori dan wilayah berasal dari `groupby`, bukan dari tebakan model. Kunci evidence berikut akan dipakai untuk menelusuri setiap temuan dan rekomendasi.

**Tanya kelas:** Apakah profit yang rendah membuktikan bahwa diskon *menyebabkan* kerugian? Data agregat ini hanya menunjukkan hubungan yang perlu diperiksa lagi.

In [ ]:
def build_evidence(frame: pd.DataFrame) -> tuple[dict[str, int | float | str], pd.DataFrame, pd.DataFrame]:
    total_sales = float(frame["sales"].sum())
    if total_sales <= 0:
        raise ValueError("Total sales harus positif untuk menghitung profit margin.")

    category = (
        frame.groupby("category", as_index=False)
        .agg(sales=("sales", "sum"), profit=("profit", "sum"))
        .sort_values(["profit", "category"], ascending=[True, True])
    )
    region = (
        frame.groupby("region", as_index=False)
        .agg(sales=("sales", "sum"), profit=("profit", "sum"))
        .sort_values(["profit", "region"], ascending=[True, True])
    )

    facts: dict[str, int | float | str] = {
        "row_count": len(frame),
        "total_sales": round(total_sales, 2),
        "total_profit": round(float(frame["profit"].sum()), 2),
        "profit_margin": round(float(frame["profit"].sum()) / total_sales, 4),
        "average_discount": round(float(frame["discount"].mean()), 4),
        "lowest_profit_category": str(category.iloc[0]["category"]),
        "lowest_profit_region": str(region.iloc[0]["region"]),
    }
    for _, row in category.iterrows():
        facts[f"category_profit.{row['category']}"] = round(float(row["profit"]), 2)
    for _, row in region.iterrows():
        facts[f"region_profit.{row['region']}"] = round(float(row["profit"]), 2)
    return facts, category, region


evidence, category_summary, region_summary = build_evidence(data)
assert round(float(category_summary["profit"].sum()), 2) == evidence["total_profit"]
assert round(float(region_summary["profit"].sum()), 2) == evidence["total_profit"]
print(json.dumps(evidence, ensure_ascii=False, indent=2))
display(category_summary, region_summary)

## 4. Output visual: pola untuk manajer

Grafik memakai tabel agregasi yang sama dengan evidence. Periksa label, satuan, dan apakah urutan kategori membantu pembaca melihat profit terendah.

**Prediksi:** Kategori dan wilayah mana akan muncul di sisi profit terendah? Verifikasi dengan nilai `category_profit.*` dan `region_profit.*` di atas.

In [ ]:
category_chart = px.bar(
    category_summary,
    x="category", y="profit", color="profit",
    title="Profit per Kategori Produk — Juli 2026",
    labels={"category": "Kategori", "profit": "Profit (satuan dataset)"},
    hover_data=["sales"],
    color_continuous_scale="RdYlGn",
)
category_chart.update_layout(coloraxis_showscale=False)
category_chart.show()

region_chart = px.bar(
    region_summary,
    x="region", y="profit", color="profit",
    title="Profit per Wilayah — Juli 2026",
    labels={"region": "Wilayah", "profit": "Profit (satuan dataset)"},
    hover_data=["sales"],
    color_continuous_scale="RdYlGn",
)
region_chart.update_layout(coloraxis_showscale=False)
region_chart.show()

## 5. Kontrak narasi dan rekomendasi

Contoh kalimat lemah: **“Diskon tinggi menyebabkan kerugian Furniture.”** Grafik dan agregasi belum membuktikan sebab. Minta model menulis dari evidence saja, lalu periksa hasilnya.

LangChain akan meminta output sesuai bentuk Pydantic. Skema di bawah sengaja hanya menentukan **bentuk dasar** (field, tipe, pilihan aksi). Pemeriksaan jumlah temuan, kunci evidence, dan kewajiban review dilakukan lagi oleh kode bisnis. Validasi bentuk **tidak** membuktikan bahwa sebuah kalimat benar.

In [ ]:
class Finding(BaseModel):
    model_config = ConfigDict(extra="forbid")

    statement: str
    evidence_keys: list[str]


class ActionRecommendation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    action: Literal[
        "review_low_profit_category", "review_regional_performance", "monitor_only"
    ]
    priority: Literal["low", "medium", "high"]
    owner: Literal["sales_ops", "finance_ops", "regional_manager"]
    reason: str
    evidence_keys: list[str]
    requires_human_review: bool


class AIReport(BaseModel):
    model_config = ConfigDict(extra="forbid")

    executive_summary: str
    key_findings: list[Finding]
    recommendations: list[str]
    limitations: list[str]
    action: ActionRecommendation


def validate_evidence_keys(keys: list[str], facts: dict[str, int | float | str]) -> None:
    if not keys:
        raise ValueError("Minimal satu evidence key diperlukan.")
    unknown = sorted(set(keys) - set(facts))
    if unknown:
        raise ValueError(f"Evidence key tidak dikenal: {', '.join(unknown)}")


def validate_business_rules(report: AIReport, facts: dict[str, int | float | str]) -> None:
    if not 2 <= len(report.key_findings) <= 3:
        raise ValueError("Laporan harus memuat 2–3 temuan.")
    if not report.executive_summary.strip() or not report.recommendations or not report.limitations:
        raise ValueError("Ringkasan, rekomendasi, dan batasan wajib terisi.")
    for finding in report.key_findings:
        if not finding.statement.strip():
            raise ValueError("Pernyataan temuan tidak boleh kosong.")
        validate_evidence_keys(finding.evidence_keys, facts)
    validate_evidence_keys(report.action.evidence_keys, facts)
    if not report.action.reason.strip():
        raise ValueError("Alasan aksi tidak boleh kosong.")
    if report.action.action != "monitor_only" and not report.action.requires_human_review:
        raise ValueError("Aksi tindak lanjut wajib menunggu review manusia.")
    if report.action.action == "monitor_only" and report.action.priority != "low":
        raise ValueError("monitor_only harus memiliki priority=low.")


def action_status(action: ActionRecommendation) -> dict[str, str | bool]:
    return {
        "status": (
            "logged_no_external_action" if action.action == "monitor_only"
            else "awaiting_human_approval"
        ),
        "executed": False,
        "action": action.action,
    }

### Jeda: uji pagar pengaman tanpa API

Dua payload sengaja salah di bawah menguji lapisan berbeda: kunci evidence yang tidak ada dan nama aksi di luar daftar. Tidak ada laporan pengganti atau aksi sungguhan.

**Prediksi:** Kesalahan mana ditolak oleh aturan bisnis, dan mana ditolak oleh Pydantic?

In [ ]:
try:
    validate_evidence_keys(["category_profit.Bulan"], evidence)
except ValueError as exc:
    print("Ditolak oleh aturan evidence:", exc)

invalid_action = {
    "action": "issue_refund",
    "priority": "high",
    "owner": "finance_ops",
    "reason": "Contoh aksi yang tidak diizinkan.",
    "evidence_keys": ["total_profit"],
    "requires_human_review": False,
}
try:
    ActionRecommendation.model_validate(invalid_action)
except ValidationError:
    print("Ditolak oleh Pydantic: action tidak ada dalam daftar izin.")

assert action_status(
    ActionRecommendation(
        action="review_low_profit_category", priority="medium", owner="sales_ops",
        reason="Periksa kategori berprofit paling rendah sebelum menentukan tindakan.",
        evidence_keys=["lowest_profit_category"], requires_human_review=True,
    )
)["executed"] is False

## 6. Satu panggilan LangChain untuk laporan terstruktur

Lihat tiga komponen di sel berikut: `ChatPromptTemplate` membatasi tugas, `ChatOpenAI` memilih model, dan `with_structured_output` menetapkan bentuk respons. Hanya **satu** pemanggilan `chain.invoke` yang memakai API. `OPENAI_MODEL` boleh diubah di environment; jika kosong, modelnya `gpt-4o-mini`.

Jika key, akses model, jaringan, atau batas penggunaan bermasalah, sel ini berhenti dengan pesan ringkas. Evidence dan grafik di atas tetap dapat ditinjau; tidak dibuat narasi atau rekomendasi palsu.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY belum tersedia di environment atau .env repository.")

model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip() or "gpt-4o-mini"
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Anda analis retail yang menulis Bahasa Indonesia untuk manajer. "
        "Gunakan HANYA evidence JSON yang diberikan; jangan menghitung ulang, "
        "mengarang angka, atau menyatakan sebab-akibat yang belum dibuktikan. "
        "Buat tepat 2–3 key_findings; tiap temuan mengutip minimal satu evidence_keys "
        "yang persis ada di JSON. Sertakan ringkasan singkat, 1–2 rekomendasi, "
        "dan sedikitnya satu keterbatasan. Pilih satu aksi yang diizinkan. "
        "Jika aksi bukan monitor_only, requires_human_review harus true. "
        "Jika monitor_only, priority harus low. Jangan menyatakan aksi telah dijalankan."
    ),
    ("human", "Tulis laporan dari evidence berikut:\n{evidence_json}"),
])
model = ChatOpenAI(
    model=model_name, temperature=0, max_tokens=1500, timeout=90, max_retries=0,
)
chain = prompt | model.with_structured_output(
    AIReport, method="json_schema", strict=True,
)

try:
    report = chain.invoke({"evidence_json": json.dumps(evidence, ensure_ascii=False)})
    if not isinstance(report, AIReport):
        raise ValueError("Respons model tidak berupa AIReport.")
    validate_business_rules(report, evidence)
except Exception as exc:
    raise RuntimeError(
        f"Laporan model tidak tersedia ({type(exc).__name__}). "
        "Periksa key, akses model, jaringan, dan aturan validasi; jangan lanjutkan aksi."
    ) from None

print(f"Laporan tervalidasi dari {model_name}. Aksi eksternal: tidak ada.")

## 7. Audit klaim sebelum presentasi

Cocokkan tiap kalimat temuan dengan nilai evidence yang ditampilkan. Kode memastikan kuncinya ada, tetapi **manusia tetap harus memeriksa** apakah kalimat benar, proporsional, dan tidak membuat klaim sebab-akibat.

**Tanya kelas:** Adakah angka atau kata seperti “karena” yang tidak didukung oleh evidence?

In [ ]:
audit_rows = [
    {
        "temuan": finding.statement,
        "evidence": "; ".join(f"{key} = {evidence[key]}" for key in finding.evidence_keys),
    }
    for finding in report.key_findings
]
display(pd.DataFrame(audit_rows))

## 8. Sajikan output naratif dan actionable

Pembaca bisnis mendapat ringkasan dan temuan; sistem downstream hanya mendapat rekomendasi terstruktur. Status di bawah selalu menunjukkan `executed: false`. Dalam demo ini tidak ada fungsi eksekusi aksi.

In [ ]:
def render_report(result: AIReport, facts: dict[str, int | float | str]) -> str:
    lines = ["# Laporan Insight Retail", "", "## Ringkasan", "", result.executive_summary,
             "", "## Temuan", ""]
    for finding in result.key_findings:
        lines.append(f"- {finding.statement}")
        lines.append("  - Evidence: " + "; ".join(
            f"`{key}` = {facts[key]}" for key in finding.evidence_keys
        ))
    lines.extend(["", "## Rekomendasi", ""])
    lines.extend(f"- {item}" for item in result.recommendations)
    lines.extend(["", "## Batasan", ""])
    lines.extend(f"- {item}" for item in result.limitations)
    lines.extend(["", "> Rekomendasi belum disetujui atau dijalankan."])
    return "\n".join(lines)


display(Markdown(render_report(report, evidence)))
print("Rekomendasi terstruktur:")
print(json.dumps(report.action.model_dump(), ensure_ascii=False, indent=2))
print("Status aman:")
print(json.dumps(action_status(report.action), ensure_ascii=False, indent=2))
assert action_status(report.action)["executed"] is False

## 9. Diskusi penutup

1. Mengapa agregasi untuk grafik tidak diserahkan kepada model?
2. Pilih satu kalimat dari laporan. Kunci evidence apa yang mendukungnya? Apakah nilai yang disebut benar?
3. Apa yang lolos dari validasi skema tetapi masih dapat menyesatkan pengguna?
4. Siapa yang berwenang menyetujui aksi, dan data tambahan apa yang diperlukan?

**Checkpoint instruktur:** visual, narasi, dan JSON ditampilkan inline; temuan dapat ditelusuri; `executed` selalu `false`. Sel model adalah satu-satunya langkah yang membutuhkan API key. Setelah demo, gunakan **Kernel → Restart & Clear Outputs** sebelum membagikan notebook agar respons API dan data runtime tidak ikut tersimpan.